<a href="https://colab.research.google.com/github/2303A51434/natural-language-processing/blob/main/lab_test_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 60.9 MB/s eta 0:00:00


In [15]:
import numpy as np
import time

# -------------------------------------
# START TIMER
# -------------------------------------
start_time = time.time()

# -----------------------------
# Step 1: Load GloVe Manually
# -----------------------------
def load_glove(file_path):
    embeddings = {}
    expected_vector_size = None
    with open(file_path, 'r', encoding="utf8") as f:
        for line_num, line in enumerate(f, 1):
            try:
                parts = line.strip().split()
                if len(parts) < 2:
                    print(f"Skipping line {line_num} (less than 2 parts): {line.strip()}")
                    continue
                word = parts[0]
                vector = np.array(parts[1:], dtype=float)

                if expected_vector_size is None:
                    expected_vector_size = vector.shape[0]

                if vector.shape[0] != expected_vector_size:
                    print(f"Skipping line {line_num} ('{line.strip()}') due to inconsistent vector size: Expected {expected_vector_size}, got {vector.shape[0]}")
                    continue

                embeddings[word] = vector

            except ValueError as e:
                print(f"Skipping malformed line {line_num}: '{line.strip()}' due to error: {e}")
                continue
            except Exception as e:
                print(f"An unexpected error occurred on line {line_num}: '{line.strip()}' due to error: {e}")
                continue
    return embeddings

print("Loading GloVe embeddings...")
glove = load_glove("glove.6B.100d.txt")
print("Vocabulary size:", len(glove))


# -----------------------------
# Step 2: Cosine Similarity
# -----------------------------
def cosine_sim(a, b):
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return np.dot(a, b) / (norm_a * norm_b)


# -----------------------------
# Step 3: Explore Embeddings
# -----------------------------
def get_word_vector(word, embeddings_dict, word_list_name=""):
    if word in embeddings_dict:
        return embeddings_dict[word]
    else:
        print(f"'{word}' not found in vocabulary. (for {word_list_name})")
        return None

# 1. Vector of a word
print("\nVector of 'india':")
india_vec = get_word_vector('india', glove)
if india_vec is not None:
    print(india_vec[:10])

# 2. Similarity
print("\nSimilarity between 'king' and 'queen':")
king_vec = get_word_vector('king', glove)
queen_vec = get_word_vector('queen', glove)
if king_vec is not None and queen_vec is not None:
    print(cosine_sim(king_vec, queen_vec))

# 3. Analogy: king - man + woman
print("\nAnalogy: king - man + woman = ?")
positive_words = ['king', 'woman']
negative_words = ['man']

ready = True
analogy_vecs = {}

for w in positive_words + negative_words:
    vec = get_word_vector(w, glove, "analogy")
    if vec is None:
        ready = False
        break
    analogy_vecs[w] = vec

if ready:
    target = analogy_vecs['king'] - analogy_vecs['man'] + analogy_vecs['woman']
    best_word = ""
    best_score = -1

    for word, vec in glove.items():
        if word not in positive_words + negative_words:
            sim = cosine_sim(target, vec)
            if sim > best_score:
                best_word = word
                best_score = sim

    print("Result:", best_word)
else:
    print("Skipping analogy due to missing words.")

# 4. Similar words
print("\nWords similar to 'computer':")
computer_vec = get_word_vector('computer', glove)
if computer_vec is not None:
    best_word = ""
    best_score = -1
    for word, vec in glove.items():
        if word != 'computer':
            sim = cosine_sim(computer_vec, vec)
            if sim > best_score:
                best_word = word
                best_score = sim
    print(f"Most similar: {best_word} (score: {best_score:.4f})")

# 5. Odd one out
print("\nOdd one out:")
words = ['car', 'bus', 'train', 'banana']

vecs = {}
ready = True
for w in words:
    v = get_word_vector(w, glove, "odd-one-out")
    if v is None:
        ready = False
        break
    vecs[w] = v

if ready:
    scores = {}
    for w in words:
        others = [vecs[o] for o in words if o != w]
        avg_vec = np.mean(others, axis=0)
        scores[w] = cosine_sim(vecs[w], avg_vec)

    print(min(scores, key=scores.get))
else:
    print("Skipping odd one out due to missing words.")

# -------------------------------------
# END TIMER
# -------------------------------------
end_time = time.time()
print("\nTotal Execution Time:", end_time - start_time, "seconds")


Loading GloVe embeddings...
Skipping malformed line 1225: 'standard -0.13416 0.47697 0.45242 0.2767 -0.25912 0.01633 -0.04325 -0.050398 -0.44321 0.77802 0.25418 -0.43745 -0.35909 0.0296 0.23188 -0.2735 0.52555 -0.42301 0.33815 -0.46337 0.21922 -0.57964 0.46917 -0.10207 0.79651 -0.72937 -0.22785 -0.75203 -0.80478 0.46363 0.5083 0.62078 -0.66816 0.020804 1.0387 0.91916 0.32015 0.3477 0.44449 -0.15881 0.01747 -0.4890.034001 0.067387 0.97522 0.21002 -0.87387 -1.0978 -0.99056 0.020703 -0.2838 -0.19559 0.57675 -0.096618 0.16953 -0.50773 -0.30557 -0.16253' due to error: could not convert string to float: '-0.4890.034001'
Skipping line 2446 ('younes -0.37012 -1.1411 0.61125 -0.054828 0.44163 0.13727 0.48774 1.0789 0.025863 0.32249 0.5877 -0.15802 -1.4068 -0.18964 1.2291 0.42099 -0.53159 -0.097692 -0.47694 -0.24974 -0.50608 0.12651 0.60884 0.87562 0.77391 0.28134 0.40662 -1.2031 0.30121 -0.31508 -0.79456 -0.54967 0.71344 0.34772 -0.69697 -0.60614 -0.11508 0.82891 -0.2429 0.38198 -0.45863 0.4577

In [12]:
import time
from gensim.models import Word2Vec

# ------------------------------------------------
# START TIMER
# ------------------------------------------------
start_time = time.time()

# ------------------------------------------------
# 1. Sample Dataset (Small Corpus → Very Fast)
# ------------------------------------------------
sentences = [
    "machine learning is fun",
    "deep learning uses neural networks",
    "word embeddings capture meaning",
    "python is a programming language",
    "data science includes statistics",
    "artificial intelligence uses data"
]

# Preprocessing
corpus = [s.split() for s in sentences]

# ------------------------------------------------
# 2. Train Word2Vec Model
# ------------------------------------------------
model = Word2Vec(
    sentences=corpus,
    vector_size=50,
    window=3,
    min_count=1,
    workers=1
)

# ------------------------------------------------
# 3. Explore Word Embeddings
# ------------------------------------------------
print("Vocabulary size:", len(model.wv.key_to_index))

print("\nVector for 'learning':")
print(model.wv['learning'])

print("\nMost similar words to 'data':")
print(model.wv.most_similar("data"))

print("\nSimilarity between 'machine' and 'learning':")
print(model.wv.similarity("machine", "learning"))

print("\nOdd one out among ['python', 'java', 'banana']:")
print(model.wv.doesnt_match(['python', 'java', 'banana']))

# ------------------------------------------------
# END TIMER
# ------------------------------------------------
end_time = time.time()
print("\nTotal Execution Time:", end_time - start_time, "seconds")



Vocabulary size: 22

Vector for 'learning':
[ 1.56351421e-02 -1.90203730e-02 -4.11062239e-04  6.93839323e-03
 -1.87794445e-03  1.67635437e-02  1.80215668e-02  1.30730132e-02
 -1.42324204e-03  1.54208085e-02 -1.70686692e-02  6.41421322e-03
 -9.27599426e-03 -1.01779103e-02  7.17923651e-03  1.07406788e-02
  1.55390287e-02 -1.15330126e-02  1.48667218e-02  1.32509926e-02
 -7.41960062e-03 -1.74912829e-02  1.08749345e-02  1.30195115e-02
 -1.57510047e-03 -1.34197120e-02 -1.41718509e-02 -4.99412045e-03
  1.02865072e-02 -7.33047491e-03 -1.87401194e-02  7.65347946e-03
  9.76895820e-03 -1.28571270e-02  2.41711619e-03 -4.14975407e-03
  4.88066689e-05 -1.97670180e-02  5.38400887e-03 -9.50021297e-03
  2.17529293e-03 -3.15244915e-03  4.39334614e-03 -1.57631524e-02
 -5.43436781e-03  5.32639725e-03  1.06933638e-02 -4.78302967e-03
 -1.90201886e-02  9.01175756e-03]

Most similar words to 'data':
[('neural', 0.2704731225967407), ('capture', 0.21063268184661865), ('a', 0.16714175045490265), ('meaning', 0.15